***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 4 章：可见度空间](4_0_introduction.ipynb)
    * 上一节：[4.2 基线及其空间表示](4_2_the_baseline_and_its_representation_in_space.ipynb)
    * 下一节：[4.4 可见度函数](4_4_the_visibility_function.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 4.3 二元干涉仪<a id='visibility:sec:two_element_interferometer'></a>

二元干涉仪是理解可见度的最小模型。两面天线接收来自同一片天空的电磁波，相关器把两路电压相乘并做时间平均，输出一个复数。这个复数不是源的总功率，而是天空亮度在某个空间频率上的投影。第 4.2 节定义的投影基线决定了这个空间频率的位置。

本节从两个层次建立这个结论。第一层是普通波动干涉：两列相干波叠加时，光程差产生相位差，进而产生明暗条纹。第二层是射电干涉仪的接收问题：天线并不让两个天体彼此“发生干涉”，而是把同一方向到达两面天线的电场做互相关。不同天空方向彼此不相干，所以总响应是各方向贡献的线性积分。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

# Figure 4.3.1: two-element geometry.
fig, ax = plt.subplots(figsize=(9.2, 5.4))
ax.set_aspect('equal')
ax.axis('off')
R1 = np.array([-1.8, 0.0])
R2 = np.array([1.8, 0.0])
ax.scatter([R1[0], R2[0]], [R1[1], R2[1]], s=180, color='#d95f02', zorder=4)
ax.text(R1[0], R1[1]-0.25, 'R1', ha='center', va='top', fontsize=13)
ax.text(R2[0], R2[1]-0.25, 'R2', ha='center', va='top', fontsize=13)
ax.annotate('', xy=R2, xytext=R1, arrowprops=dict(arrowstyle='<->', lw=2.0, color='#333333'))
ax.text(0, -0.22, 'baseline b', ha='center', va='top', fontsize=12)

angle = np.deg2rad(55)
s = np.array([np.cos(angle), np.sin(angle)])
n = np.array([-np.sin(angle), np.cos(angle)])
for offset in np.linspace(-3.2, 3.2, 6):
    center = np.array([0.0, 0.35]) + offset*n
    p1 = center - 3.2*n
    p2 = center + 3.2*n
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='#8da0cb', lw=1.2, alpha=0.8)
ax.annotate('', xy=-1.1*s + np.array([0, 3.9]), xytext=np.array([0, 3.9]),
            arrowprops=dict(arrowstyle='-|>', lw=2.2, color='#1b7837'))
ax.text(-1.3*s[0], 3.9-1.3*s[1]+0.05, 'incoming direction s', color='#1b7837', fontsize=12, ha='center')
ax.plot([R1[0], R1[0]+0.9*s[0]], [R1[1], R1[1]+0.9*s[1]], color='#444444', lw=1.1, ls='--')
ax.plot([R2[0], R2[0]+0.9*s[0]], [R2[1], R2[1]+0.9*s[1]], color='#444444', lw=1.1, ls='--')
ax.text(0.55, 1.05, r'geometric delay: $\tau_g=(\mathbf{b}\cdot\mathbf{s})/c$', fontsize=12)
ax.set_xlim(-3.6, 3.6)
ax.set_ylim(-1.0, 4.4)
fig.savefig(fig_dir / 'two_element_geometry.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Figure 4.3.2: complex fringe and visibility components.
l = np.linspace(-0.55, 0.55, 900)
u0 = 8.0
real_fringe = np.cos(2*np.pi*u0*l)
imag_fringe = -np.sin(2*np.pi*u0*l)
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), constrained_layout=True)
axes[0].plot(l, real_fringe, color='#2b6cb0', lw=1.8, label='real: cos fringe')
axes[0].plot(l, imag_fringe, color='#c53030', lw=1.8, label='imag: -sin fringe')
axes[0].axhline(0, color='0.75', lw=0.8)
axes[0].set_xlabel('direction cosine l')
axes[0].set_ylabel('fringe response')
axes[0].set_title('Complex fringe for one baseline')
axes[0].legend(frameon=False, loc='upper right')

u = np.linspace(0, 18, 700)
source_symmetric = np.exp(-2j*np.pi*u*(-0.11)) + np.exp(-2j*np.pi*u*(0.11))
source_asymmetric = 1.0*np.exp(-2j*np.pi*u*(-0.11)) + 0.55*np.exp(-2j*np.pi*u*(0.19))
axes[1].plot(u, source_symmetric.real, color='#2b6cb0', lw=1.5, label='symmetric Re')
axes[1].plot(u, source_symmetric.imag, color='#2b6cb0', lw=1.5, ls='--', label='symmetric Im')
axes[1].plot(u, source_asymmetric.real, color='#c53030', lw=1.5, label='asymmetric Re')
axes[1].plot(u, source_asymmetric.imag, color='#c53030', lw=1.5, ls='--', label='asymmetric Im')
axes[1].axhline(0, color='0.75', lw=0.8)
axes[1].set_xlabel('baseline coordinate u')
axes[1].set_ylabel('visibility')
axes[1].set_title('Real and imaginary visibility encode sky symmetry')
axes[1].legend(frameon=False, fontsize=9, ncol=2)
fig.savefig(fig_dir / 'complex_correlator_visibility.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Figure 4.3.3: bandwidth decorrelation and delay tracking.
x = np.linspace(-4.0, 4.0, 1000)
amp = np.abs(np.sinc(x))
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(x, amp, color='#4c78a8', lw=2.0)
ax.fill_between(x, 0, amp, color='#4c78a8', alpha=0.12)
ax.axvline(0, color='#2f855a', lw=2.0, ls='--', label='tracked phase center')
ax.axvline(1.7, color='#c53030', lw=1.8, ls=':', label='residual delay')
ax.set_xlabel(r'residual delay in bandwidth units  $\Delta\nu\,\tau$')
ax.set_ylabel('correlation amplitude')
ax.set_title('Bandwidth decorrelation follows a sinc envelope')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.savefig(fig_dir / 'bandwidth_delay_tracking.png', dpi=180, bbox_inches='tight')
plt.close(fig)


### 4.3.1 从相干叠加到相位差<a id='visibility:sec:coherent_superposition'></a>

先考虑普通波动干涉。若两列同频率、相干的标量波在空间中某点 $P$ 相遇，可以写成

$$
s_1(P,t)=s_{01}\cos(\omega t-k r_1+\varphi_{01}),
$$

$$
s_2(P,t)=s_{02}\cos(\omega t-k r_2+\varphi_{02}),
$$

其中 $r_1$ 和 $r_2$ 是两列波从各自源点到 $P$ 的传播距离，$k=2\pi/\lambda$ 是波数。若两列波振幅相同、初始相位相同，则叠加后有

$$
s_1+s_2=2s_0\cos\left(\frac{\Delta\Phi}{2}\right)
\cos\left(\omega t-k\frac{r_1+r_2}{2}+\varphi_0\right),
$$

其中

$$
\Delta\Phi = k(r_2-r_1)=\frac{2\pi}{\lambda}\Delta L.
$$

当 $\Delta L=n\lambda$ 时，两列波相长干涉；当 $\Delta L=(n+1/2)\lambda$ 时，两列波相消干涉。条纹图样本质上就是等相位差曲面的集合。


![二元干涉仪几何](figures/two_element_geometry.png)

**图 4.3.1** 二元干涉仪的几何延迟。对来自方向 $\mathbf{s}$ 的平面波，两面天线之间的到达时间差为 $\tau_g=(\mathbf{b}\cdot\mathbf{s})/c$。


### 4.3.2 接收问题：天体之间不互相干涉<a id='visibility:sec:receiving_interference'></a>

上面的相干叠加图像很直观，但射电干涉仪面对的物理情况稍有不同。天线接收的是来自天空许多方向的随机电磁场。不同天体、甚至同一扩展源中相距足够远的不同区域，通常彼此不相干；它们的电场不会像实验室中两台相干发射机那样在空间中形成稳定明暗条纹。

干涉仪真正做的是另一件事：对同一方向到达两面天线的电场进行互相关。来自方向 $\mathbf{s}$ 的窄带平面波到达两面天线时存在几何延迟

$$
\tau_g(\mathbf{s})=\frac{\mathbf{b}\cdot\mathbf{s}}{c}.
$$

若把第 1 面天线作为参考，两路复电压可近似写成

$$
v_1(t;\mathbf{s}) = a_1(\mathbf{s})E(t),
$$

$$
v_2(t;\mathbf{s}) = a_2(\mathbf{s})E(t-\tau_g),
$$

其中 $a_i(\mathbf{s})$ 表示天线方向响应。对窄带信号，延迟等价于相位因子，因此相关器对该方向的响应含有

$$
\exp\left[-2\pi i\nu\tau_g(\mathbf{s})\right]
=
\exp\left[-2\pi i\frac{\mathbf{b}\cdot\mathbf{s}}{\lambda}\right].
$$

这就是基线作为空间频率的来源：方向 $\mathbf{s}$ 的亮度被一个由 $\mathbf{b}/\lambda$ 决定的复指数条纹加权。


### 4.3.3 相关器与复可见度<a id='visibility:sec:complex_correlator'></a>

相关器输出的是两路电压乘积的时间平均。若使用复基带表示，理想相关器可写成

$$
C_{12}=\left\langle v_1(t)v_2^*(t)\right\rangle_t.
$$

对单个点源，$C_{12}$ 与源强成正比，并带有上面的几何相位因子。对连续天空，来自不同方向的贡献彼此不相干，时间平均会消去不同方向之间的交叉项，只留下各方向自相关项的线性积分。于是

$$
C_{12}=\int A_{12}(\mathbf{s}) I_\nu(\mathbf{s})
\exp\left[-2\pi i\frac{\mathbf{b}\cdot\mathbf{s}}{\lambda}\right]d\Omega,
$$

其中 $I_\nu(\mathbf{s})$ 是天空比强度，$A_{12}=a_1a_2^*$ 是两天线方向响应的乘积。若相关器还对相位中心 $\mathbf{s}_0$ 做相位跟踪，就会把相位中心方向的相位因子除掉，得到

$$
C_{12}=\int A_{12}(\mathbf{s}) I_\nu(\mathbf{s})
\exp\left[-2\pi i\frac{\mathbf{b}\cdot(\mathbf{s}-\mathbf{s}_0)}{\lambda}\right]d\Omega.
$$

这个复数就是可见度的物理雏形。余弦相关器测量实部，正弦相关器测量虚部；两者合在一起，才能同时记录条纹响应的幅度和相位。


![复相关器与可见度](figures/complex_correlator_visibility.png)

**图 4.3.2** 左图显示一条基线对应的复条纹，实部和虚部分别是相差 $90^\circ$ 的余弦与正弦响应。右图显示对称和非对称双点源的可见度：天空不对称时，虚部携带不可忽略的信息。


### 4.3.4 从方向向量到 $(u,v,w)$ 形式<a id='visibility:sec:uvw_phase'></a>

第 4.2 节已经把基线写成以波长为单位的 $(u,v,w)$。第 3.4 节则把天空方向写成相对于相位中心的方向余弦 $(l,m,n)$。两者相乘时，几何相位自然变成

$$
\frac{\mathbf{b}\cdot(\mathbf{s}-\mathbf{s}_0)}{\lambda}
= ul+vm+w(n-1).
$$

因此，相关器输出可以写成

$$
V(u,v,w)=\iint \frac{A(l,m)I_\nu(l,m)}{n}
\exp\left[-2\pi i\left(ul+vm+w(n-1)\right)\right] dl\,dm.
$$

这里把 $C_{12}$ 改写为 $V$，是为了强调它作为可见度函数样本的角色。若视场足够小，使得 $n\approx 1$ 且 $w(n-1)$ 可忽略，同时暂时忽略主波束变化，则上式退化为二维傅里叶变换：

$$
V(u,v)\approx\iint I_\nu(l,m)e^{-2\pi i(ul+vm)}dl\,dm.
$$

这条近似关系将在第 4.6 节中更系统地推导。此处先保留一个核心认识：一条基线的相关器输出，就是天空亮度在一个空间频率点上的复投影。


### 4.3.5 带宽效应与时延跟踪<a id='visibility:sec:delay_tracking'></a>

上面的推导默认窄带信号，因而可以把延迟写成单一相位因子。真实观测有有限带宽 $\Delta\nu$。若某方向相对于相位中心仍有残余延迟 $\tau$，不同频率上的相位因子会在频带内旋转；频率平均后，相关幅度会被削弱。对理想矩形频带，有

$$
\frac{1}{\Delta\nu}\int_{-\Delta\nu/2}^{+\Delta\nu/2}
 e^{-2\pi i\nu'\tau}d\nu'
=\operatorname{sinc}(\Delta\nu\tau),
$$

其中 $\operatorname{sinc}(x)=\sin(\pi x)/(\pi x)$。这就是带宽去相关（bandwidth decorrelation）的基本来源。

时延跟踪的作用，是在仪器或后处理里补偿相位中心方向的几何延迟，使相位中心附近的信号在频带内保持相干。离相位中心越远，残余延迟越大，带宽去相关越明显；这也是宽带宽、长基线和宽视场观测中必须认真处理延迟模型的原因。


![带宽去相关与时延跟踪](figures/bandwidth_delay_tracking.png)

**图 4.3.3** 有限带宽下，残余延迟会使不同频率的相位旋转相互抵消，相关幅度按 sinc 包络下降。时延跟踪把相位中心方向移动到包络中心。


### 4.3.6 本节要点

二元干涉仪的核心不是让两个天体互相干涉，而是让两面天线接收到的同一电磁场做互相关。几何延迟 $\tau_g=\mathbf{b}\cdot\mathbf{s}/c$ 在窄带下变成相位因子，基线 $\mathbf{b}/\lambda$ 因而成为空间频率。对连续天空，不同方向贡献线性叠加，得到的复相关量就是可见度样本。余弦和正弦相关分别对应实部和虚部；时延跟踪和带宽处理则保证有限频带内的相关不会被几何延迟洗掉。下一节将把这些单条基线、单个时刻的样本提升为完整的可见度函数。


***

* 下一节：[4.4 可见度函数](4_4_the_visibility_function.ipynb)
